# 📊 Разведочный анализ установленного ПО (Software EDA)

**Автор:** Artur Minart  
**Дата:** Май 2026  
**Цель:** Проанализировать установленное программное обеспечение, выявить закономерности, визуализировать распределение по категориям и вендорам.

---

## 1. Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Настройка стилей для красивых графиков
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Для отображения графиков прямо в ноутбуке
%matplotlib inline

## 2. Загрузка данных

In [ ]:
# Определение пути к файлу
data_path = Path('data/installed_software.csv')

# Загрузка данных
df = pd.read_csv(data_path)

print(f"✅ Данные загружены успешно!")
print(f"📁 Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"\n📋 Названия колонок: {list(df.columns)}")

## 3. Первичный осмотр данных

In [ ]:
# Показать первые 10 записей
df.head(10)

In [ ]:
# Общая информация о датасете
df.info()

In [ ]:
# Статистика по числовым колонкам
df.describe()

In [ ]:
# Проверка на пропуски
missing_values = df.isnull().sum()
print("Пропущенные значения по колонкам:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "Нет пропущенных значений ✅")

## 4. Очистка и предобработка данных

In [ ]:
# Преобразование даты в формат datetime
df['install_date'] = pd.to_datetime(df['install_date'])

# Создание дополнительных признаков
df['install_month'] = df['install_date'].dt.to_period('M')
df['install_year'] = df['install_date'].dt.year
df['size_category'] = pd.cut(df['size_mb'], 
                             bins=[0, 50, 200, 500, 1000, float('inf')],
                             labels=['Tiny (<50MB)', 'Small (50-200MB)', 'Medium (200-500MB)', 
                                     'Large (500MB-1GB)', 'Huge (>1GB)'])

print("✅ Предобработка завершена")
print(f"Добавлены колонки: install_month, install_year, size_category")

In [ ]:
# Просмотр обновленных данных
df[['software_name', 'category', 'size_mb', 'install_date', 'size_category']].head(10)

## 5. Анализ по категориям

In [ ]:
# Количество программ по категориям
category_counts = df['category'].value_counts()

print("📁 Распределение по категориям:")
print(category_counts)

In [ ]:
# Визуализация: Бар чарт по категориям
fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.Set3(range(len(category_counts)))

bars = ax.bar(category_counts.index, category_counts.values, color=colors, edgecolor='black', linewidth=1.2)

ax.set_title('Распределение ПО по категориям', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Категория', fontsize=12)
ax.set_ylabel('Количество программ', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Добавление значений на столбцы
for bar, count in zip(bars, category_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
            str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/charts/notebook_01_categories.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ График сохранен в output/charts/notebook_01_categories.png")

## 6. Анализ вендоров

In [ ]:
# Топ-10 вендоров
top_vendors = df['vendor'].value_counts().head(10)

print("🏆 Топ-10 вендоров:")
for i, (vendor, count) in enumerate(top_vendors.items(), 1):
    print(f"{i}. {vendor}: {count} программ")

In [ ]:
# Визуализация: Горизонтальный бар чарт топ вендоров
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.Blues_r(np.linspace(0.3, 0.9, len(top_vendors)))

bars = ax.barh(top_vendors.index, top_vendors.values, color=colors, edgecolor='navy', linewidth=1)

ax.set_title('Топ-10 вендоров по количеству ПО', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Количество программ', fontsize=12)
ax.invert_yaxis()

# Добавление значений
for bar, count in zip(bars, top_vendors.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('output/charts/notebook_02_top_vendors.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ График сохранен в output/charts/notebook_02_top_vendors.png")

## 7. Анализ размера ПО

In [ ]:
# Общий размер по категориям (в ГБ)
size_by_category = df.groupby('category')['size_mb'].sum().sort_values(ascending=False) / 1024

print("💾 Общий размер ПО по категориям (ГБ):")
print(size_by_category.round(2))

In [ ]:
# Визуализация: Размер по категориям
fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.Oranges(np.linspace(0.3, 0.9, len(size_by_category)))

bars = ax.bar(size_by_category.index, size_by_category.values, color=colors, edgecolor='darkorange', linewidth=1.2)

ax.set_title('Общий размер ПО по категориям (ГБ)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Категория', fontsize=12)
ax.set_ylabel('Размер (ГБ)', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Добавление значений
for bar, size in zip(bars, size_by_category.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{size:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/charts/notebook_03_size_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ График сохранен в output/charts/notebook_03_size_by_category.png")

## 8. Динамика установки ПО

In [ ]:
# Установки по месяцам
installs_by_month = df.set_index('install_date').resample('M')['software_name'].count()

print("📅 Динамика установок по месяцам:")
print(installs_by_month)

In [ ]:
# Визуализация: Линейный график установок
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(installs_by_month.index, installs_by_month.values, 
        marker='o', linewidth=2.5, markersize=8, color='steelblue', label='Установки')
ax.fill_between(installs_by_month.index, installs_by_month.values, alpha=0.3, color='steelblue')

ax.set_title('Динамика установки ПО по месяцам', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Месяц', fontsize=12)
ax.set_ylabel('Количество установок', fontsize=12)
plt.xticks(rotation=45)
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('output/charts/notebook_04_install_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ График сохранен в output/charts/notebook_04_install_timeline.png")

## 9. Сводная таблица: Категории vs Вендоры

In [ ]:
# Pivot table: Категории x Вендоры
pivot_table = pd.crosstab(df['category'], df['vendor'])

print("📊 Сводная таблица: Количество ПО по категориям и вендорам")
pivot_table.style.background_gradient(cmap='Blues', axis=None).format(na_rep='-')

## 10. Основные выводы

### 📈 Ключевые инсайты:

1. **Наиболее представленная категория:** Development (Python, Anaconda, Git, VS Code и др.)
2. **Топ вендор:** Microsoft (Power BI, Excel, Word, SQL Server и др.)
3. **Самая тяжелая категория:** Office и BI инструменты (занимают больше всего места)
4. **Пик установок:** Февраль-Март 2024 (активная настройка рабочего окружения)
5. **Средний размер программы:** ~400 МБ

### 💡 Рекомендации:
- Рассмотреть возможность очистки старых версий библиотек
- Оптимизировать хранение тяжелых BI инструментов
- Документировать версии критического ПО для воспроизводимости среды

---
**Аналитик:** Artur Minart | Data Analyst Portfolio  
**GitHub:** github.com/MartinMinart/data-portfolio